# Data Cleaning

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load dataset

df = pd.read_csv("../data/walmart_products_sample_5000.csv")

### Fixing GTIN Issues

In [3]:
# Extract only GTINs with length < 12
short_gtins = df.loc[df["Gtin"].astype(str).str.len() < 12, "Gtin"]

print(short_gtins)

0       28478129474
1         403371886
3         244644791
6         333261538
9       22808261216
           ...     
4982      719178714
4985    90852910237
4986    36000193176
4994    38000157165
4996      557230999
Name: Gtin, Length: 1851, dtype: int64


In [4]:
# Convert GTIN to string and get lengths
df["gtin_length"] = df["Gtin"].astype(str).str.len()

# Count occurrences of each GTIN length as percentages
length_distribution_percent = df["gtin_length"].value_counts(normalize=True).sort_index() * 100

print(length_distribution_percent)

gtin_length
6      0.02
8      1.72
9     13.00
10     0.22
11    22.06
12    62.98
Name: proportion, dtype: float64


#### Technically, standard GTINs used in retail are usually 8, 12, 13, or 14 digits:

**GTIN-8**	Small products / limited space barcodes

**GTIN-12**	UPC (mostly in North America)

**GTIN-13**	EAN (mostly international)

**GTIN-14**	Shipping containers, pallets

Converting 9 and 11 digit GTINs as 12 digit because there is a high possibility that leading zeroes are truncated from these GTINs while exporting the file. However further analysis is needed.

In [5]:
# Convert GTIN to string to preserve leading zeros
df["Gtin"] = df["Gtin"].astype(str)

# Function to pad GTIN to 12 digits if it's 9 or 11 digits
def pad_gtin(Gtin):
    if len(Gtin) == 9 or len(Gtin) == 11:
        return Gtin.zfill(12)  # add leading zeros
    else:
        return Gtin  # leave others unchanged

# Apply the function to the original GTIN column
df["Gtin"] = df["Gtin"].apply(pad_gtin)

print(df['Gtin'])

0       028478129474
1       000403371886
2       641275538562
3       000244644791
4       858098007094
            ...     
4995    722195376653
4996    000557230999
4997    191344004967
4998    746839700970
4999    667544768300
Name: Gtin, Length: 5000, dtype: object


In [6]:
# Convert GTIN column to string
df["gtin_str"] = df["Gtin"].astype(str)

# Check for scientific notation
scientific_gtins = df[df["gtin_str"].str.contains("e", case=False)]

print(scientific_gtins)

Empty DataFrame
Columns: [Gtin, Brand, Product Name, Description, List Price, Package Size, Category, Product Url, gtin_length, gtin_str]
Index: []


### Fixing Brand Issues

In [7]:
# Strip leading/trailing spaces, keeping NaNs
df["Brand"] = df["Brand"].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Filter rows where brand is empty or NaN
empty_brand_records = df[df["Brand"].isna() | (df["Brand"] == "")]

print(empty_brand_records)

              Gtin Brand                                       Product Name  \
52    689182221535   NaN  WP000-PT 1713 1713 Strips Test s Panels Incl M...   
123   070273500026   NaN  CREE Q5 1000LM LED Spiked Mace Baseball Bat Lo...   
170   661666674082   NaN  Worth Softball Face Mask Guard Black Metal Siz...   
332   000359855179   NaN             (2 Pack) Maggi Spaetzle 10.5 Oz. Pouch   
407   000467651302   NaN  Dermatological E45 Cream, Treatment for Dry Sk...   
...            ...   ...                                                ...   
4641  765448960582   NaN  Skin Shaver Corn Cuticle Cutter Remover Rasp P...   
4771  657789553366   NaN  Juicy Fruit Starburst Strawberry Sugarfre Gum,...   
4847  613635314659   NaN  Protective Hip Pad Padded Shorts Skiing Skatin...   
4872  745559541375   NaN  (5 pack) Marijuana (THC) Urine Drug Testing Di...   
4989  747960394014   NaN                          White Swan Christmas Sled   

                                            Descrip

### Product Name Issues

In [8]:
# Strip leading/trailing spaces, keeping NaNs
df["Product Name"] = df["Product Name"].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Filter rows where brand is empty or NaN
empty_product_name_records = df[df["Product Name"].isna() | (df["Product Name"] == "")]

print(empty_product_name_records)

Empty DataFrame
Columns: [Gtin, Brand, Product Name, Description, List Price, Package Size, Category, Product Url, gtin_length, gtin_str]
Index: []


In [9]:
# Brand and Product Names cleaning function

import re

def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'[^A-Za-z0-9 ]+', '', text)  # remove special characters
        text = text.strip()                          # remove spaces
        text = text.title()                          # capitalize each word
        return text
    return text

# Apply to brand and product
df["Brand"] = df["Brand"].apply(clean_text)
df["Product Name"] = df["Product Name"].apply(clean_text)

print(df)

              Gtin                     Brand  \
0     028478129474           Crosman Archery   
1     000403371886                     Dilwe   
2     641275538562  Little Things Mean A Lot   
3     000244644791               College Inn   
4     858098007094                     Cremo   
...            ...                       ...   
4995  722195376653                  Toolworx   
4996  000557230999                 St Tropez   
4997  191344004967               Pure Garden   
4998  746839700970         After Market Ezgo   
4999  667544768300          Bath  Body Works   

                                           Product Name  \
0              Centerpoint Sentinel Long Bow Set Aby215   
1     Dilwe 7W E27 Smart Bluetooth Rgb Color Changin...   
2     Boys Polyrayon Gabardine Bib With Button Accen...   
3            2 Pack College Inn Chicken Bold Stock 32Oz   
4     Cremo Astonishingly Superior Shave Cream Bourb...   
...                                                 ...   
4995      

###  Fixing Description 

In [10]:
# Text to remove
text_to_remove = "We aim to show you accurate product information. Manufacturers, suppliers and others provide what you see here, and we have not verified it. See our disclaimer |"

# Remove the text from description column
df["Description"] = df["Description"].str.replace(text_to_remove, "", regex=False).str.strip()

print(df['Description'])

0       Equipped with durable, heavy weight fiberglass...
1       Description:  This remote control smart light ...
2       Boys soft polyrayon gabardine christening bib ...
3       Chicken Bold Stock. A small amount of glutamat...
4       Our shave creams are made of super-slick molec...
                              ...                        
4995    Measures 8 inch Two-sided rasp Refine skin to ...
4996    Achieve a natural, healthy looking tan with th...
4997    This solar mosquito bug zapper light by pure g...
4998    Spark plug, Champion RC12YC. For E-Z-GO gas 20...
4999    Fragrance A cozy, irresistible scent of Intoxi...
Name: Description, Length: 5000, dtype: object


### Size Extraction

In [11]:
# Regex pattern to match common sizes
size_pattern = r'(\d+\.?\d*\s?(ml|l|g|kg|oz|lb|cm|mm|in))'

# Extract size and store in new column
df['Size'] = df['Product Name'].str.lower().str.extract(size_pattern, expand=False)[0]

# Remove size from product name
df['Product Name'] = df['Product Name'].str.replace(size_pattern, '', regex=True).str.strip()

print(df['Size'])

0         NaN
1         NaN
2         NaN
3        32oz
4         6oz
        ...  
4995      NaN
4996    67 oz
4997      NaN
4998    480 g
4999      NaN
Name: Size, Length: 5000, dtype: object


In [12]:
# Count nulls in size column

null_count = df['Size'].isna().sum()
print("Number of nulls in size column:", null_count)

Number of nulls in size column: 3486


In [17]:
# Create a new column "taxonomy"

# Copy values from "category" to "taxonomy"

df['Taxonomy'] = df['Category'].copy()

print(df['Taxonomy'])


0       Sports & Outdoors | Outdoor Sports | Hunting |...
1        Household Essentials | Light Bulbs | Smart Bulbs
2                   Baby | Feeding | Bibs and Burp Cloths
3       Food | Meal Solutions, Grains & Pasta | Canned...
4          Personal Care | Personal Care by Brand | Cremo
                              ...                        
4995                      Health | Foot Care | Foot Files
4996    Personal Care | Sun Care | Self-Tanners & Bron...
4997    Household Essentials | Pest Control | All Pest...
4998    Sports & Outdoors | Sports | Golf Equipment | ...
4999    Personal Care | Bath & Body | Body Lotions & C...
Name: Taxonomy, Length: 5000, dtype: object


In [18]:
# Keep only first catgeory in "Category" column

df['Category'] = df['Category'].str.split('|').str[0].str.strip()

print(df['Category'])

0          Sports & Outdoors
1       Household Essentials
2                       Baby
3                       Food
4              Personal Care
                ...         
4995                  Health
4996           Personal Care
4997    Household Essentials
4998       Sports & Outdoors
4999           Personal Care
Name: Category, Length: 5000, dtype: object


In [20]:
print(df.columns)

Index(['Gtin', 'Brand', 'Product Name', 'Description', 'List Price',
       'Package Size', 'Category', 'Product Url', 'gtin_length', 'gtin_str',
       'Size', 'Taxonomy'],
      dtype='object')


In [ ]:
# Remove below columns and store result in new DataFrame
cleaned_df = df.drop(columns=['Package Size', 'Product Url', 'gtin_length', 'gtin_str'])

# Export to CSV (without index column)
cleaned_df.to_csv("../data/walmart_products_cleaned_5000.csv", index=False)

print("Cleaned CSV saved with 5,000 records!") 